# Notebook 57 — Coefficient Manifold Learning and Structured Meta-Law Dynamics

Notebook 56 showed that state-space coordinate transforms do not improve the global law.
This suggests that the shared field is already expressed in the right state variables `(r,c)`,
and that the real structure lives in how the **coefficient vector changes across regimes**.

We keep the fixed field template

\[
g(r,c;\beta)=\beta_0+\beta_1 c+\beta_2 r+\beta_3 c^3+\beta_4 r^2+\beta_5 r c^2
\]

and move the modeling target to the coefficient manifold itself.

## Main goals

1. Characterize the geometry of the coefficient manifold.
2. Identify coefficient families and metadata-aligned transport directions.
3. Fit structured coefficient transport laws:
   - additive
   - additive + selected interactions
   - low-rank manifold-coordinate model
4. Evaluate harder holdout blocks using:
   - coefficient RMSE
   - field RMSE
   - trajectory RMSE

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Fixed coefficient template per regime

TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(TERM_NAMES, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def predict_with_beta(sub, beta):
    return design_template(sub) @ beta

def trajectory_gap(sub, beta_ref, beta_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i+1] - cgrid[i])
            x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
            g = float(np.clip(x @ beta, -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    errs = []
    for r0 in r0s:
        t_ref = integrate(beta_ref, r0)
        t_cmp = integrate(beta_cmp, r0)
        errs.append(math.sqrt(mean_squared_error(t_ref, t_cmp)))
    return float(np.mean(errs))

In [ ]:
# Regime table and metadata design matrices

group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(group_cols):
    if len(sub) < 30:
        continue
    beta, pred, stats = fit_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    }
    row.update(stats)
    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)
Y_coef = coef_df[COEF_COLS].copy()

display(coef_df.head())

## Coefficient manifold geometry

In [ ]:
# PCA spectrum and manifold coordinates

scaler = StandardScaler()
Y_std = scaler.fit_transform(Y_coef.to_numpy(dtype=float))
pca_full = PCA(random_state=42)
pca_full.fit(Y_std)

explained = pca_full.explained_variance_ratio_
cumexplained = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(np.arange(1, len(explained) + 1), explained, marker="o")
axes[0].set_title("Coefficient manifold scree plot")
axes[0].set_xlabel("component")
axes[0].set_ylabel("explained variance ratio")

axes[1].plot(np.arange(1, len(cumexplained) + 1), cumexplained, marker="o")
axes[1].set_title("Cumulative explained variance")
axes[1].set_xlabel("component")
axes[1].set_ylabel("cumulative explained variance")

plt.tight_layout()
plt.show()

pca2 = PCA(n_components=2, random_state=42)
Z2 = pca2.fit_transform(Y_std)
coef_df["pc1"] = Z2[:, 0]
coef_df["pc2"] = Z2[:, 1]

display(coef_df[["regime_id", "pc1", "pc2"]].head())

In [ ]:
# Coefficient manifold colored by metadata

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, col in zip(axes.ravel(), ["forcing_mode", "system", "flow_mode", "k"]):
    if col == "k":
        sc = ax.scatter(coef_df["pc1"], coef_df["pc2"], c=coef_df["k"], s=45)
        plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    else:
        for value in sorted(coef_df[col].astype(str).unique()):
            sub = coef_df[coef_df[col].astype(str) == value]
            ax.scatter(sub["pc1"], sub["pc2"], label=value, s=45)
        ax.legend()
    ax.set_title(f"Coefficient manifold colored by {col}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.show()

## Coefficient covariance and coefficient families

In [ ]:
# Correlation structure and term-family view

corr = Y_coef.corr()

plt.figure(figsize=(8, 6))
plt.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar(label="correlation")
plt.title("Coefficient correlation heatmap")
plt.tight_layout()
plt.show()

display(corr)

In [ ]:
# Pairwise coefficient scatter: key families

pairs = [("coef_r", "coef_r^2"), ("coef_r", "coef_r c^2"), ("coef_c", "coef_c^3"), ("coef_1", "coef_c")]
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for ax, (xcol, ycol) in zip(axes.ravel(), pairs):
    for value in sorted(coef_df["forcing_mode"].astype(str).unique()):
        sub = coef_df[coef_df["forcing_mode"].astype(str) == value]
        ax.scatter(sub[xcol], sub[ycol], label=value, s=40)
    ax.set_xlabel(xcol)
    ax.set_ylabel(ycol)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Metadata-aligned coefficient transport directions

In [ ]:
# Tangent-direction estimates from simple grouped means

def mean_coef(group_col):
    return coef_df.groupby(group_col)[COEF_COLS].mean()

forcing_means = mean_coef("forcing_mode")
system_means = mean_coef("system")
flow_means = mean_coef("flow_mode")

display(forcing_means)
display(system_means)
display(flow_means)

In [ ]:
# Tangent arrows in PCA space for forcing and system means

forcing_mean_pc = pca2.transform(scaler.transform(forcing_means.to_numpy()))
system_mean_pc = pca2.transform(scaler.transform(system_means.to_numpy()))
flow_mean_pc = pca2.transform(scaler.transform(flow_means.to_numpy()))

plt.figure(figsize=(8, 6))
plt.scatter(coef_df["pc1"], coef_df["pc2"], alpha=0.25, s=25)

for name, pt in zip(forcing_means.index.tolist(), forcing_mean_pc):
    plt.scatter(pt[0], pt[1], s=120, marker="x")
    plt.text(pt[0], pt[1], f"forcing:{name}")

for name, pt in zip(system_means.index.tolist(), system_mean_pc):
    plt.scatter(pt[0], pt[1], s=120, marker="^")
    plt.text(pt[0], pt[1], f"system:{name}")

plt.title("Mean transport directions in coefficient manifold")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

## Structured coefficient transport models

In [ ]:
# Design matrices for structured transport laws

def build_additive_X(df_in, columns=None):
    X = pd.get_dummies(df_in[["system", "task", "forcing_mode", "flow_mode"]], drop_first=False)
    X["k"] = df_in["k"].astype(float).values
    X["k2"] = df_in["k"].astype(float).values ** 2
    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)
    return X

def build_interaction_X(df_in, columns=None):
    X = build_additive_X(df_in).copy()

    # selected interactions
    sf = df_in["system"].astype(str) + "__x__" + df_in["forcing_mode"].astype(str)
    ff = df_in["forcing_mode"].astype(str) + "__x__" + df_in["flow_mode"].astype(str)

    X_sf = pd.get_dummies(sf, prefix="sf")
    X_ff = pd.get_dummies(ff, prefix="ff")
    X_fk = pd.get_dummies(df_in["forcing_mode"].astype(str), prefix="fk").multiply(
        df_in["k"].astype(float).to_numpy().reshape(-1, 1)
    )

    X = pd.concat([X, X_sf, X_ff, X_fk], axis=1)
    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)
    return X

def fit_predict_multioutput_ridge(X_train, Y_train, X_test, alpha=1.0):
    model = Ridge(alpha=alpha)
    model.fit(np.asarray(X_train, float), np.asarray(Y_train, float))
    return model.predict(np.asarray(X_test, float)), model

def fit_predict_manifold_model(X_train, Y_train, X_test, n_components=2, alpha=1.0):
    scaler = StandardScaler()
    Ytr_std = scaler.fit_transform(np.asarray(Y_train, float))
    pca = PCA(n_components=n_components, random_state=42)
    Ztr = pca.fit_transform(Ytr_std)

    model = Ridge(alpha=alpha)
    model.fit(np.asarray(X_train, float), Ztr)
    Zhat = model.predict(np.asarray(X_test, float))

    Yhat_std = pca.inverse_transform(Zhat)
    Yhat = scaler.inverse_transform(Yhat_std)
    return Yhat, model, pca, scaler

## Hard-block evaluation

In [ ]:
hard_block_masks = {
    "k_extrapolate_high": coef_df["k"].astype(float) == 7.0,
    "k_extrapolate_low": coef_df["k"].astype(float) == 3.0,
    "system_holdout::entropy": coef_df["system"].astype(str) == "entropy",
    "system_holdout::unevenness": coef_df["system"].astype(str) == "unevenness",
}

rows = []

for block_name, test_mask in hard_block_masks.items():
    train_mask = ~test_mask

    df_train = coef_df.loc[train_mask].reset_index(drop=True)
    df_test = coef_df.loc[test_mask].reset_index(drop=True)

    Y_train = df_train[COEF_COLS]
    Y_test = df_test[COEF_COLS]

    X_add_train = build_additive_X(df_train)
    X_add_test = build_additive_X(df_test, columns=X_add_train.columns)

    X_int_train = build_interaction_X(df_train)
    X_int_test = build_interaction_X(df_test, columns=X_int_train.columns)

    Yhat_add, add_model = fit_predict_multioutput_ridge(X_add_train, Y_train, X_add_test, alpha=1.0)
    Yhat_int, int_model = fit_predict_multioutput_ridge(X_int_train, Y_train, X_int_test, alpha=1.0)
    Yhat_man, man_model, man_pca, man_scaler = fit_predict_manifold_model(X_add_train, Y_train, X_add_test, n_components=2, alpha=1.0)

    # flat baseline from earlier style
    Yhat_flat, flat_model = fit_predict_multioutput_ridge(X_add_train, Y_train, X_add_test, alpha=10.0)

    for i, rid in enumerate(df_test["regime_id"].tolist()):
        sub = regime_subsets[rid]
        beta_true = Y_test.iloc[i].to_numpy(dtype=float)
        beta_flat = Yhat_flat[i]
        beta_add = Yhat_add[i]
        beta_int = Yhat_int[i]
        beta_man = Yhat_man[i]

        y_emp = sub["predicted_flow"].to_numpy(dtype=float)

        pred_flat = predict_with_beta(sub, beta_flat)
        pred_add = predict_with_beta(sub, beta_add)
        pred_int = predict_with_beta(sub, beta_int)
        pred_man = predict_with_beta(sub, beta_man)

        rows.append({
            "block": block_name,
            "regime_id": rid,
            "coef_rmse_flat": math.sqrt(mean_squared_error(beta_true, beta_flat)),
            "coef_rmse_additive": math.sqrt(mean_squared_error(beta_true, beta_add)),
            "coef_rmse_interaction": math.sqrt(mean_squared_error(beta_true, beta_int)),
            "coef_rmse_manifold": math.sqrt(mean_squared_error(beta_true, beta_man)),

            "field_rmse_flat": math.sqrt(mean_squared_error(y_emp, pred_flat)),
            "field_rmse_additive": math.sqrt(mean_squared_error(y_emp, pred_add)),
            "field_rmse_interaction": math.sqrt(mean_squared_error(y_emp, pred_int)),
            "field_rmse_manifold": math.sqrt(mean_squared_error(y_emp, pred_man)),

            "traj_rmse_flat": trajectory_gap(sub, beta_true, beta_flat),
            "traj_rmse_additive": trajectory_gap(sub, beta_true, beta_add),
            "traj_rmse_interaction": trajectory_gap(sub, beta_true, beta_int),
            "traj_rmse_manifold": trajectory_gap(sub, beta_true, beta_man),
        })

compare_df = pd.DataFrame(rows)
display(compare_df.head())

In [ ]:
summary_df = compare_df.groupby("block")[[
    "coef_rmse_flat", "coef_rmse_additive", "coef_rmse_interaction", "coef_rmse_manifold",
    "field_rmse_flat", "field_rmse_additive", "field_rmse_interaction", "field_rmse_manifold",
    "traj_rmse_flat", "traj_rmse_additive", "traj_rmse_interaction", "traj_rmse_manifold",
]].mean().reset_index()

display(summary_df)

In [ ]:
# Model comparison plots on harder blocks

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
x = np.arange(len(summary_df))
w = 0.20

axes[0].bar(x - 1.5*w, summary_df["coef_rmse_flat"], width=w, label="flat")
axes[0].bar(x - 0.5*w, summary_df["coef_rmse_additive"], width=w, label="additive")
axes[0].bar(x + 0.5*w, summary_df["coef_rmse_interaction"], width=w, label="interaction")
axes[0].bar(x + 1.5*w, summary_df["coef_rmse_manifold"], width=w, label="manifold")
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary_df["block"], rotation=30, ha="right")
axes[0].set_title("Coefficient RMSE")
axes[0].legend()

axes[1].bar(x - 1.5*w, summary_df["field_rmse_flat"], width=w, label="flat")
axes[1].bar(x - 0.5*w, summary_df["field_rmse_additive"], width=w, label="additive")
axes[1].bar(x + 0.5*w, summary_df["field_rmse_interaction"], width=w, label="interaction")
axes[1].bar(x + 1.5*w, summary_df["field_rmse_manifold"], width=w, label="manifold")
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary_df["block"], rotation=30, ha="right")
axes[1].set_title("Field RMSE")
axes[1].legend()

axes[2].bar(x - 1.5*w, summary_df["traj_rmse_flat"], width=w, label="flat")
axes[2].bar(x - 0.5*w, summary_df["traj_rmse_additive"], width=w, label="additive")
axes[2].bar(x + 0.5*w, summary_df["traj_rmse_interaction"], width=w, label="interaction")
axes[2].bar(x + 1.5*w, summary_df["traj_rmse_manifold"], width=w, label="manifold")
axes[2].set_xticks(x)
axes[2].set_xticklabels(summary_df["block"], rotation=30, ha="right")
axes[2].set_title("Trajectory RMSE")
axes[2].legend()

plt.tight_layout()
plt.show()

## Representative manifold transport cases

In [ ]:
# Pick regime where interaction model improves most over flat on trajectory RMSE

rep_idx = int(np.argmax(compare_df["traj_rmse_flat"] - compare_df["traj_rmse_interaction"]))
rep = compare_df.iloc[rep_idx]
regime_id = rep["regime_id"]
block_name = rep["block"]

test_mask = coef_df["regime_id"] == regime_id
train_mask = ~test_mask
df_train = coef_df.loc[train_mask].reset_index(drop=True)
df_test = coef_df.loc[test_mask].reset_index(drop=True)

Y_train = df_train[COEF_COLS]
Y_test = df_test[COEF_COLS]

X_add_train = build_additive_X(df_train)
X_add_test = build_additive_X(df_test, columns=X_add_train.columns)
X_int_train = build_interaction_X(df_train)
X_int_test = build_interaction_X(df_test, columns=X_int_train.columns)

Yhat_add, _ = fit_predict_multioutput_ridge(X_add_train, Y_train, X_add_test, alpha=1.0)
Yhat_int, _ = fit_predict_multioutput_ridge(X_int_train, Y_train, X_int_test, alpha=1.0)
Yhat_man, _, _, _ = fit_predict_manifold_model(X_add_train, Y_train, X_add_test, n_components=2, alpha=1.0)

beta_true = Y_test.iloc[0].to_numpy(dtype=float)
beta_add = Yhat_add[0]
beta_int = Yhat_int[0]
beta_man = Yhat_man[0]

coef_compare = pd.DataFrame({
    "term": COEF_COLS,
    "true": beta_true,
    "additive": beta_add,
    "interaction": beta_int,
    "manifold": beta_man,
})
display(coef_compare)

In [ ]:
# Representative trajectory comparison

sub = regime_subsets[regime_id]
cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
rmin, rmax = sub["residual"].min(), sub["residual"].max()
cgrid = np.linspace(cmin, cmax, 45)
r0s = np.linspace(np.quantile(sub["residual"], 0.1), np.quantile(sub["residual"], 0.9), 8)
flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

def integrate_beta(beta, r0):
    r = float(np.clip(r0, rmin, rmax))
    traj = [r]
    for i in range(len(cgrid) - 1):
        c = cgrid[i]
        dc = float(cgrid[i+1] - cgrid[i])
        x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
        g = float(np.clip(x @ beta, -flow_cap, flow_cap))
        r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
        traj.append(r)
    return np.array(traj)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
titles = ["True", "Additive", "Interaction", "Manifold"]
betas = [beta_true, beta_add, beta_int, beta_man]

for ax, title, beta in zip(axes, titles, betas):
    for r0 in r0s:
        ax.plot(cgrid, integrate_beta(beta, r0))
    ax.set_title(title)
    ax.set_xlabel("condition coordinate c")
axes[0].set_ylabel("residual")
fig.suptitle(f"Representative regime: {regime_id}", y=1.03)
plt.tight_layout()
plt.show()

## Final verdicts

In [ ]:
def verdict_row(row):
    best_coef = min(row["coef_rmse_additive"], row["coef_rmse_interaction"], row["coef_rmse_manifold"])
    best_field = min(row["field_rmse_additive"], row["field_rmse_interaction"], row["field_rmse_manifold"])
    best_traj = min(row["traj_rmse_additive"], row["traj_rmse_interaction"], row["traj_rmse_manifold"])

    if best_traj < row["traj_rmse_flat"]:
        if row["traj_rmse_interaction"] <= row["traj_rmse_additive"] and row["traj_rmse_interaction"] <= row["traj_rmse_manifold"]:
            return "interaction transport best"
        if row["traj_rmse_manifold"] <= row["traj_rmse_additive"]:
            return "manifold transport best"
        return "additive transport best"
    return "flat model remains best"

summary_df["verdict"] = summary_df.apply(verdict_row, axis=1)
display(summary_df)

## Working conclusion

Notebook 57 treats regime variation as structured motion on a low-dimensional coefficient manifold.

A strong result is:
- coefficient variation is low-rank
- forcing and system organize major manifold directions
- additive or low-order interaction transport laws improve hard-block prediction
- manifold-aware coefficient transport outperforms ad hoc state-space transforms

If that pattern holds on your real exports, the next notebook should be:

**Notebook 58 — symbolic transport law for coefficients**